# E15 (expanded) — Threshold-based decision analysis over lead times (absorbs E16)

**Protocol:** `DECISIONS.md` — the "PRE-REGISTRATION: expanded threshold-based decision analysis"
(2026-09-18), Sidh's design decisions (2026-09-19), and the implementation AMENDMENT (2026-09-19).
All were written before any full-grid result existed. **Spec:** `EXPERIMENT_PLAN.md` §E15
(expansion). The matched-budget E15 results (`reports/05_decision_cost.html`) stand unchanged.

- **Rule:** alert on an event iff its score ≥ t.
- **Grid T, per horizon (EXTENDED, Sidh 2026-09-20):** the sorted, deduplicated **union** of
  (a) the 5th–95th percentiles of pooled point predictions, (b) the same percentiles of the pooled
  values of **every validated bound** (all methods, both sidednesses), and (c) −6 — all percentiles
  on the training pool's **internal validation split**. The official test set never selects the
  grid. Component (b) fixes the ceiling artifact flagged on 2026-09-19: with (a) alone every
  percentile fell below −6, the grid's maximum *was* −6, and 62.3% of bound threshold selections
  pinned to it.
- **Lead times (Q-METH-04, resolved):** **2-day and 3-day, both on the official test set.** Each
  horizon has its own derived configuration and its own fresh hyperparameter search.
- **Populations:**
  - `common_across_horizons` — the official-test events predictable at both horizons. This is the
    **lead-time comparison's population**.
  - `horizon_full` — each horizon's own full set. The 2-day full set matches E15.
- **Lens (D1):** true high-risk events are primary; the whole population is secondary. Missed
  high-risk events is the lead number.
- **Arms:** the point prediction; split and weighted conformal, both as a one-sided bound and a
  two-sided upper edge (two-sided arms retained, Sidh 2026-09-19); CQR in both; the E8 Bayesian
  bound in both. Persistence's one-sided bounds are excluded with disclosure.
- **Prediction P1**, made precise by Corollary 4 of Proposition 1:
  - **P1a:** for a bound point + Q, the operating point at a fixed threshold shifts by exactly Q;
  - **P1b:** the translation class's operating locus is unchanged;
  - **P1c:** its unrestricted cost optimum is unchanged.
- **Statistics (D4):** descriptive only, with event-level bootstrap CIs.

> **Caveat carried by every result.** One-sided upper bounds under-cover on the official test set
> (the E11 diagnostic); persistence's one-sided bound is degenerate (Gate 2) and excluded here.
> Bounds and point predictions share one grid defined in point-prediction space.

In [ ]:
# Papermill parameters. SMOKE=True renders the reduced configuration (pre-registration §9):
# pipeline validation and timing ONLY - its numbers are not findings.
SMOKE = False

In [ ]:
# --- Setup + provenance (invariant I4) ------------------------------------------------------
import json, subprocess, sys
from datetime import datetime, timezone

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from kelvins_conformal.config import REPO_ROOT, load_config
from kelvins_conformal.models import decision_runner as DR
from kelvins_conformal.models import threshold_runner as TR
from kelvins_conformal.reporting import write_table_atomic

cfg = load_config()
TA = cfg.threshold_analysis
PREFIX = "e15c_smoke_" if SMOKE else "e15c_"
FIGDIR = cfg.path("figures_dir"); FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = cfg.path("tables_dir"); TABDIR.mkdir(parents=True, exist_ok=True)

def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT),
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return "UNAVAILABLE"

def save_table(df, name):
    write_table_atomic(df, TABDIR / f"{PREFIX}{name}.csv"); print(f"saved: reports/tables/{PREFIX}{name}.csv")

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIGDIR / f"{PREFIX}{name}.{ext}", dpi=160, bbox_inches="tight", facecolor=fig.get_facecolor())
    print(f"saved: reports/figures/{PREFIX}{name}.png|pdf")

RUN_KWARGS = dict(seeds=list(TA.smoke_seeds), percentiles=list(TA.smoke_grid_percentiles)) if SMOKE else {}
PROVENANCE = {"experiment_ids": ["E15"], "analysis": "expanded threshold-based decision analysis (E16 merged)",
              "smoke": SMOKE, "git_commit_sha": git_sha(), "config_hash": cfg.config_hash,
              "run_kwargs": RUN_KWARGS, "horizons_days": list(TA.horizons_days),
              "executed_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0]}
print(json.dumps(PROVENANCE, indent=2))
if SMOKE:
    print("\n*** SMOKE RUN: pipeline validation and timing only - numbers are NOT findings (pre-registration §9) ***")

# Chart chrome + categorical slots 1-7 of the dataviz reference palette (light mode), fixed order,
# validated with its palette script. Colour follows the arm (method x sidedness), never its rank.
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
ARM_STYLE = {
    ("point", "point"): ("point prediction", "#2a78d6"),
    (DR.E10, "upper"): ("split conformal, one-sided", "#eb6834"),
    (DR.E10_TWO, "two"): ("split conformal, two-sided upper edge", "#1baf7a"),
    (DR.E11, "upper"): ("weighted conformal, one-sided", "#eda100"),
    (DR.E11_TWO, "two"): ("weighted conformal, two-sided upper edge", "#e87ba4"),
    (DR.E12, "upper"): ("CQR, one-sided", "#008300"),
    (DR.E12_TWO, "two"): ("CQR, two-sided upper edge", "#4a3aa7"),
    (DR.E8, "upper"): ("Bayesian (E8), one-sided", "#008300"),
    (DR.E8_TWO, "two"): ("Bayesian (E8), two-sided upper edge", "#4a3aa7"),
}
TRANSLATION_METHODS = {DR.E10, DR.E11, DR.E10_TWO, DR.E11_TWO}
plt.rcParams.update({"axes.edgecolor": AXIS, "axes.labelcolor": INK2, "xtick.color": MUTED,
                     "ytick.color": MUTED, "text.color": INK, "axes.titlecolor": INK})

## 1. Run

In [ ]:
RES = TR.run_threshold_analysis(cfg, **RUN_KWARGS)
meta = RES["meta"]
print(json.dumps({k: v for k, v in meta.items() if k != "caveat"}, indent=2, default=str))
for key in ("decisions", "paired_differences", "selection", "p1_checks", "operating_curves",
            "grid", "populations", "excluded_arms", "positivity", "search_integrity"):
    save_table(RES[key], key)
PRIM, CAVEAT = meta["primary_level"], meta["caveat"]
HORIZONS = meta["horizons_days"]
OP_T = float(TA.operational_thresholds[0])
dec, sel, p1 = RES["decisions"], RES["selection"], RES["p1_checks"]

## 2. Integrity, populations, exclusions, grids and timings

In [ ]:
print("Search integrity (per horizon; caches compared only at the same cutoff):")
display(RES["search_integrity"])
print("Event populations (2026-09-19 amendment, item 1):")
display(RES["populations"])
print("Excluded arms (disclosed, pre-registration §1):")
display(RES["excluded_arms"])
print("Threshold grid (2026-09-20 extension: point + bound percentiles + the operational threshold):")
G = RES["grid"]
display(G.groupby(["horizon_days", "source"]).size().unstack(fill_value=0))
display(pd.DataFrame([{
    "horizon (d)": h,
    "n thresholds": int((G["horizon_days"] == h).sum()),
    "range": f"[{G.loc[G['horizon_days'] == h, 'threshold'].min():.3f}, "
             f"{G.loc[G['horizon_days'] == h, 'threshold'].max():.3f}]",
    "n above the operational threshold": int((G.loc[G["horizon_days"] == h, "threshold"] > OP_T).sum()),
    "n from bound percentiles only": int(meta["per_horizon"][f"{h:g}d"]["n_grid_from_bound_percentiles_only"]),
} for h in HORIZONS]))
display(G.groupby("horizon_days")["threshold"].apply(lambda s: [round(v, 3) for v in s]).to_frame())
display(RES["positivity"])
print("Per-horizon meta:", json.dumps(meta["per_horizon"], indent=2))
print("Timings (s):", json.dumps(meta["timings"], indent=2))
print("Searches cached at start:", meta["searches_cached_at_start"])

## 3. Prediction P1, made precise (pre-registration §5)

These are the translation-class checks, pooled over both horizons and both populations. The
expected value is 0 for all three.
- **P1a:** counts of `bound ≥ t` equal counts of `point ≥ t − Q`.
- **P1b:** the operating locus is unchanged.
- **P1c:** the unrestricted optimal cost is unchanged.
- **Descriptive association:** Spearman correlation between Q and the fixed-threshold alert-set
  change (common population), with no test (D4).

In [ ]:
tr = p1[p1["structural_class"] == DR.TRANSLATION_OF_POINT]
P1A_MAX = float(tr["p1a_max_count_difference"].max())
P1B_MAX = float(tr["p1b_max_locus_difference_missed"].max())
P1C_MAX = float(tr[[c for c in tr.columns if c.startswith("p1c_")]].abs().max().max())
display(p1.groupby(["horizon_days", "population", "structural_class"])[
    ["offset_spread", "p1a_max_count_difference", "p1b_max_locus_difference_missed", "mean_alert_set_change_at_fixed_t"]].max().round(4))
trc = tr[tr["population"] == TR.COMMON]
RHO = stats.spearmanr(trc["offset_median_Q"], trc["mean_alert_set_change_at_fixed_t"]).statistic if len(trc) > 2 else float("nan")
print(f"Translation class: P1a max = {P1A_MAX:.3g}; P1b max = {P1B_MAX:.3g}; P1c max |cost difference| = {P1C_MAX:.3g}")
print(f"Spearman(Q, mean alert-set change at fixed t), common population, both horizons = {RHO:.3f} (descriptive)")

## 4. PRIMARY — high-risk events at the operational threshold −6, nominal 90%, common population (D1)

In [ ]:
def ci(row, name, digits=1, scale=1.0):
    return f"{row[name] * scale:.{digits}f} [{row[name + '_lo'] * scale:.{digits}f}, {row[name + '_hi'] * scale:.{digits}f}]"

def at_operational(population):
    return dec[(dec["nominal"] == PRIM) & (dec["threshold"] == OP_T) & (dec["population"] == population)
               ].sort_values(["horizon_days", "learner", "method", "sided"])

op_c = at_operational(TR.COMMON)
primary = pd.DataFrame({
    "horizon (d)": op_c["horizon_days"], "method": op_c["method"], "learner": op_c["learner"], "sided": op_c["sided"],
    "missed high-risk events [95% CI]": [ci(r, "missed_high_risk") for _, r in op_c.iterrows()],
    "recall % [95% CI]": [ci(r, "recall_high_risk", scale=100) for _, r in op_c.iterrows()],
    "alerts issued [95% CI]": [ci(r, "n_alerts") for _, r in op_c.iterrows()],
}).reset_index(drop=True)
display(primary)
save_table(primary.assign(caveat=CAVEAT), "primary_at_operational_threshold_common")
print("CAVEAT:", CAVEAT)

## 5. SECONDARY — whole population at −6, common population (D1)

In [ ]:
RATIOS = meta["cost_ratios"]
secondary = pd.DataFrame({
    "horizon (d)": op_c["horizon_days"], "method": op_c["method"], "learner": op_c["learner"], "sided": op_c["sided"],
    "unnecessary maneuvers [95% CI]": [ci(r, "unnecessary_maneuvers") for _, r in op_c.iterrows()],
    "false-positive rate % [95% CI]": [ci(r, "false_positive_rate", digits=2, scale=100) for _, r in op_c.iterrows()],
    **{f"cost {r:g}:1 [95% CI]": [ci(row, DR.dc.cost_key(r), digits=0) for _, row in op_c.iterrows()] for r in RATIOS},
}).reset_index(drop=True)
display(secondary)
save_table(secondary.assign(caveat=CAVEAT), "secondary_at_operational_threshold_common")

## 6. Lead-time comparison at −6, nominal 90%, common population (descriptive)

Both horizons are scored on the same events, so each difference reflects the lead time, not a
change of population. The per-horizon intervals are in §4 and §5.

In [ ]:
lt = op_c.pivot_table(index=["method", "learner", "sided"], columns="horizon_days",
                      values=["missed_high_risk", "unnecessary_maneuvers", "n_alerts"]).round(1)
if len(HORIZONS) == 2:
    h0, h1 = HORIZONS
    for m in ("missed_high_risk", "unnecessary_maneuvers", "n_alerts"):
        lt[(m, f"{h1:g}d - {h0:g}d")] = (lt[(m, h1)] - lt[(m, h0)]).round(1)
lt = lt.sort_index(axis=1)
display(lt)
save_table(lt.reset_index(), "lead_time_comparison_at_operational_threshold_common")

## 7. Operating curves — missed high-risk events vs. unnecessary maneuvers, point vs. bound

One figure per horizon (common population, nominal 90%), one panel per learner. Lines show each
arm's operating locus; dots mark grid thresholds and diamonds −6. Translation-class arms share the
point locus exactly (P1b, verified in §3), so they are drawn as markers on it and their shift shows
as moved markers.

In [ ]:
cur = RES["operating_curves"]
for h in HORIZONS:
    fig, axes = plt.subplots(2, 2, figsize=(13.5, 10.5), facecolor=SURFACE, sharey=True)
    c_h = cur[(cur["horizon_days"] == h) & (cur["population"] == TR.COMMON)]
    d_h = dec[(dec["horizon_days"] == h) & (dec["population"] == TR.COMMON) & (dec["nominal"] == PRIM)]
    n_hr = int(RES["populations"].set_index(["horizon_days", "population"]).loc[(h, TR.COMMON), "n_high_risk"])
    for ax, lrn in zip(axes.ravel(), DR.BASE_LEARNERS):
        ax.set_facecolor(SURFACE)
        for (method, sided), (label, colour) in ARM_STYLE.items():
            g = c_h[(c_h["learner"] == lrn) & (c_h["method"] == method) & (c_h["sided"] == sided)]
            if g.empty:
                continue
            is_point = method == "point"
            if method not in TRANSLATION_METHODS:
                ax.plot(g["unnecessary_maneuvers"], g["missed_high_risk"], color=colour,
                        lw=2.2 if is_point else 1.6, label=label, zorder=5 if is_point else 2)
            else:
                ax.plot([], [], "D", ms=7, color=colour, mec=INK, mew=0.8, label=f"{label} (markers on the point locus)")
            m = d_h[(d_h["learner"] == lrn) & (d_h["method"] == method) & (d_h["sided"] == sided)]
            ax.plot(m["unnecessary_maneuvers"], m["missed_high_risk"], "o", ms=4.5, color=colour, mec=SURFACE, mew=1.0, zorder=6)
            op = m[m["threshold"] == OP_T]
            ax.plot(op["unnecessary_maneuvers"], op["missed_high_risk"], "D", ms=8, color=colour, mec=INK, mew=0.8, zorder=7)
        ax.set_xscale("symlog", linthresh=10)
        ax.grid(True, color=GRID, lw=0.6)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        ax.set_title(lrn, loc="left", fontsize=11)
        ax.set_xlabel("unnecessary maneuvers, whole population (symmetric-log scale)")
        ax.set_ylabel(f"missed high-risk events (of {n_hr})")
        ax.legend(frameon=False, fontsize=7.5, loc="lower left")
    fig.suptitle(f"{'[SMOKE - not findings] ' if SMOKE else ''}Operating curves under the threshold rule, "
                 f"horizon {h:g} d, common population, nominal {PRIM:.0%} (dots: grid thresholds; diamonds: t = {OP_T:g})",
                 x=0.01, ha="left", fontsize=11)
    fig.text(0.01, -0.01, "Caveat: " + CAVEAT, fontsize=7.5, color=INK2, ha="left", va="top", wrap=True)
    fig.tight_layout(rect=(0, 0.02, 1, 0.96))
    save_fig(fig, f"operating_curves_h{h:g}d")
    plt.close(fig)

## 7b. Ceiling diagnostic — is the selected threshold still pinned to the grid maximum?

The artifact the extended grid targets, measured exactly the way it was measured on the original
grid: the fraction of selection rows whose chosen threshold equals that horizon's grid maximum.
On the original (point-percentile-only) grid this was **62.3% of bound selections** across the two
deployable read-outs and **54.2% of point selections**, and the grid maximum *was* the operational
threshold −6.

In [ ]:
gmax = RES["grid"].groupby("horizon_days")["threshold"].max()
sel_p = sel.assign(grid_max=sel["horizon_days"].map(gmax))
sel_p["at_ceiling"] = np.isclose(sel_p["threshold"], sel_p["grid_max"])
sel_p["arm"] = np.where(sel_p["method"] == DR.POINT, "point prediction", "bound")
DEPLOYABLE = ["selected_on_self_test", "selected_on_self_test_rule_weighted"]
ceiling = (sel_p.groupby(["readout", "arm"])["at_ceiling"].agg(["mean", "size"])
           .rename(columns={"mean": "fraction at grid maximum", "size": "n rows"}))
ceiling["fraction at grid maximum"] = (100 * ceiling["fraction at grid maximum"]).round(1)
display(ceiling)
CEIL_BOUND = 100 * float(sel_p[(sel_p["arm"] == "bound") & sel_p["readout"].isin(DEPLOYABLE)]["at_ceiling"].mean())
CEIL_POINT = 100 * float(sel_p[(sel_p["arm"] == "point prediction") & sel_p["readout"].isin(DEPLOYABLE)]["at_ceiling"].mean())
print(f"Deployable read-outs: {CEIL_BOUND:.1f}% of bound selections sit at the grid maximum "
      f"(was 62.3% on the original grid); {CEIL_POINT:.1f}% of point selections (was 54.2%).")
save_table(ceiling.reset_index(), "ceiling_diagnostic")

## 8. Cost-minimizing thresholds — does calibration shift the optimal operating point? (§4)

These use the common population at nominal 90%. The primary read-out is deployable: the
threshold is selected on each horizon's self-test split and evaluated on the official test set. Its
shift-aware variant uses the rule-weighted self-test. The two oracle read-outs are **not
achievable**; they only describe the curves.

In [ ]:
def selection_table(readout, population=TR.COMMON, level=PRIM):
    v = sel[(sel["readout"] == readout) & (sel["nominal"] == level) & (sel["population"] == population)
            ].sort_values(["horizon_days", "ratio", "learner", "method", "sided"])
    has_ci = "cost_lo" in v and v["cost_lo"].notna().any()
    return pd.DataFrame({
        "horizon (d)": v["horizon_days"], "ratio": v["ratio"], "method": v["method"], "learner": v["learner"],
        "sided": v["sided"], "threshold": v["threshold"].round(3),
        "cost [95% CI]": [ci(r, "cost", digits=0) for _, r in v.iterrows()] if has_ci else v["cost"].round(1),
        "missed": v["missed_high_risk"].round(1), "unnecessary": v["unnecessary_maneuvers"].round(1),
        "alerts": v["n_alerts"].round(1), "threshold shift vs point": v["threshold_shift_vs_point"].round(3),
        "cost change vs point [95% CI]": ([f"{r.cost_change_vs_point:+.0f} [{r.cost_change_lo:+.0f}, {r.cost_change_hi:+.0f}]"
                                           for r in v.itertuples()] if has_ci else v["cost_change_vs_point"].round(1)),
    }).reset_index(drop=True)

for readout in TR.READOUTS:
    print(f"=== {readout} (common population) ===")
    t = selection_table(readout)
    display(t)
    save_table(t.assign(caveat=CAVEAT), f"selection_{readout}_common")

## 9. Each horizon's full population (also reported, 2026-09-19 amendment item 1)

The 2-day full population is the E15 population (all official-test events).

In [ ]:
op_f = at_operational(TR.FULL)
full_tab = pd.DataFrame({
    "horizon (d)": op_f["horizon_days"], "method": op_f["method"], "learner": op_f["learner"], "sided": op_f["sided"],
    "missed high-risk events [95% CI]": [ci(r, "missed_high_risk") for _, r in op_f.iterrows()],
    "unnecessary maneuvers [95% CI]": [ci(r, "unnecessary_maneuvers") for _, r in op_f.iterrows()],
    "alerts issued": op_f["n_alerts"].round(1),
}).reset_index(drop=True)
display(full_tab)
save_table(full_tab.assign(caveat=CAVEAT), "at_operational_threshold_horizon_full")
save_table(selection_table("selected_on_self_test", population=TR.FULL).assign(caveat=CAVEAT),
           "selection_selected_on_self_test_horizon_full")

## 10. Summary — measurement only

In [ ]:
pops = RES["populations"]
if SMOKE:
    print(f"""
SMOKE RUN - PIPELINE VALIDATION AND TIMING ONLY (pre-registration §9). Numbers above are NOT findings.

 Horizons: {HORIZONS}; common population: {meta['n_common_events']} events.
 Tables produced: decisions {len(dec)}, paired differences {len(RES['paired_differences'])}, selection {len(sel)},
 P1 checks {len(p1)}, operating-curve rows {len(RES['operating_curves'])}; excluded arms {len(RES['excluded_arms'])}.
 Pipeline self-checks (structural identities): P1a max = {P1A_MAX:.3g}, P1b max = {P1B_MAX:.3g}, P1c max = {P1C_MAX:.3g}.
 Extended grid per horizon: {json.dumps({h: [meta['per_horizon'][f'{h:g}d']['n_thresholds'], meta['per_horizon'][f'{h:g}d']['n_grid_above_operational']] for h in HORIZONS})} [n thresholds, n above {OP_T:g}];
 at the grid maximum: {CEIL_BOUND:.1f}% of bound and {CEIL_POINT:.1f}% of point selections.
 Per horizon: {json.dumps(meta['per_horizon'])}
 Timings (s): {json.dumps(meta['timings'])}
 Searches cached at start: {meta['searches_cached_at_start']}
""")
else:
    print(f"""
EXPANDED E15 - THRESHOLD ANALYSIS OVER LEAD TIMES (measurement only, exactly as observed)

 Horizons {HORIZONS} d, both on the official test set; common population {meta['n_common_events']} events.
 Extended grid (point + bound percentiles, Sidh 2026-09-20) per horizon, [n thresholds, min, max, n above {OP_T:g}]:
 {json.dumps({h: [meta['per_horizon'][f'{h:g}d']['n_thresholds'], round(meta['per_horizon'][f'{h:g}d']['grid_min'], 3), round(meta['per_horizon'][f'{h:g}d']['grid_max'], 3), meta['per_horizon'][f'{h:g}d']['n_grid_above_operational']] for h in HORIZONS})}
 At the grid maximum: {CEIL_BOUND:.1f}% of bound selections, {CEIL_POINT:.1f}% of point selections
 (originally 62.3% and 54.2%, when the grid maximum was the operational threshold itself).
 P1 (translation class): P1a max = {P1A_MAX:.3g}; P1b max = {P1B_MAX:.3g}; P1c max = {P1C_MAX:.3g};
 Spearman(Q, alert-set change) = {RHO:.3f} (descriptive).
 See §4-§9 for the per-arm primary, secondary, lead-time and selection tables, each with CIs.

 CAVEAT: {CAVEAT}

 NOT DECIDED HERE: whether P1 is confirmed as a judgement; framing of the lead-time effect;
 anything in E17-E18.
""")
name = "05c_threshold_analysis_smoke_provenance.json" if SMOKE else "05c_threshold_analysis_provenance.json"
(cfg.path("reports_dir") / name).write_text(json.dumps(PROVENANCE, indent=2, default=str), encoding="utf-8")
print("provenance:", cfg.path("reports_dir") / name)